# Importing Packages

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, when, to_date, trim, to_timestamp, regexp_replace, lag, sum, date_trunc
from pyspark.sql.window import Window
from pyspark.sql.types import *
load_dotenv()

# Defining Variables

In [0]:
SILVER_SCHEMA_PATH=os.getenv('SILVER_SCHEMA_PATH')
BRONZE_SCHEMA_PATH=os.getenv('BRONZE_SCHEMA_PATH')

# Reusable Functions

In [0]:
def parse_mixed_date(df, col_name, output_col=None):
    out_col = output_col if output_col else col_name

    return df.withColumn(
        out_col,
        when(
            trim(col(col_name)).rlike(r"^\d{1,2}-\d{1,2}-\d{4}$"),
            to_date(trim(col(col_name)), "dd-MM-yyyy")
        ).when(
            trim(col(col_name)).rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"),
            to_date(trim(col(col_name)),"dd/MM/yyyy")
        )
        .when(
            trim(col(col_name)).rlike(r"^\d{4}/\d{1,2}/\d{1,2}$"),
            to_date(trim(col(col_name)), "yyyy/MM/dd")
        )
        .when(
            trim(col(col_name)).rlike(r"^\d{4}-\d{1,2}-\d{1,2}$"),
            to_date(trim(col(col_name)), "yyyy-MM-dd")
        ).otherwise(None)
    )

In [0]:
def parse_mixed_datetime(df, input_col, output_col=None):
    out_col = output_col if output_col else input_col
    return df.withColumn(
        out_col,
        when(
            trim(col(input_col)).rlike(r"^\d{1,2}-\d{1,2}-\d{4}\s\d{1,2}:\d{2}$"),
            to_timestamp(trim(col(input_col)), "dd-MM-yyyy HH:mm")
        ).when(
            trim(col(input_col)).rlike(r"^\d{1,2}/\d{1,2}/\d{4}\s\d{1,2}:\d{2}$"),
            to_timestamp(trim(col(input_col)), "d/M/yyyy H:mm")
        )
        .otherwise(None)
    )

# Date Cleaning

## Customer Table

In [0]:
customer_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_customer`""")

### Typecasting

In [0]:
customer_df = parse_mixed_date(customer_df, "account_created_date")

In [0]:
customer_df=customer_df.\
            withColumn("is_active",
            when(col("is_active")=="1",True)
            .when(col("is_active")=="0",False)
            .otherwise(None)
            )

In [0]:
customer_df=customer_df.withColumn("customer_id",col("customer_id").cast(IntegerType()))
customer_df=customer_df.withColumn("is_active",col("is_active").cast(BooleanType()))
customer_df=customer_df.withColumn("account_created_date",col("account_created_date").cast(DateType()))

### Trimming Spaces

In [0]:
customer_df=customer_df.withColumn("customer_name",trim(col("customer_name")))

### Creating Surrogate Key

In [0]:
customer_df.createOrReplaceTempView("temp")
customer_key=spark.sql("""
with cte as (
  select distinct customer_name,country,account_created_date from temp
)
select customer_name,country,account_created_date,row_number() over(order by customer_name) as customer_sk from cte 
""")

In [0]:
customer_df=customer_df.join(customer_key,["customer_name","country","account_created_date"],"left")

## Employee Table

In [0]:
employee_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_employee`""")

### Typecasting

In [0]:
employee_df = parse_mixed_date(employee_df,"last_update")
employee_df = parse_mixed_date(employee_df,"hire_date")
employee_df = employee_df.withColumn("employee_id",col("employee_id").cast(IntegerType()))

In [0]:
employee_df=employee_df.\
            withColumn("is_active_flag",
            when(col("is_active_flag")=="Yes",True)
            .when(col("is_active_flag")=="No",False)
            .otherwise(None)
            )

### Trimming Spaces

In [0]:

employee_df=employee_df.withColumn("employee_name",trim(col("employee_name")))

## Product Table

In [0]:
product_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_product`""")

### Trimming Spaces

In [0]:
product_df=product_df.withColumn("product_name",trim(col("product_name")))

### Typecasting

In [0]:
product_df = product_df.\
            withColumn("is_active",
            when(col("is_active")=="Y",True)
            .when(col("is_active")=="N",False)
            .otherwise(None)
            )

In [0]:
product_df=parse_mixed_date(product_df,"created_date")

In [0]:
product_df=product_df.withColumns(
    {
        "list_price":col("list_price").cast(IntegerType()),
        "product_id":col("product_id").cast(IntegerType())
    }
    )

### Creating Surrogate Key

In [0]:
product_df.createOrReplaceTempView("prod_temp")
product_key=spark.sql("""
with cte as (
  select distinct product_name,plan_name,billing_cycle,created_date from prod_temp
)
select product_name,plan_name,billing_cycle,created_date,row_number() over(order by product_name) as product_sk from cte 
""")

In [0]:
product_df=product_df.join(product_key,["product_name","plan_name","billing_cycle","created_date"],"left")

## Opportunity Table

In [0]:
opportunity_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_opportunity`""")

### Typecasting

In [0]:
opportunity_df=parse_mixed_datetime(opportunity_df,"created_timestamp")

In [0]:
opportunity_df=opportunity_df.withColumn("opportunity_id",col("opportunity_id").cast(IntegerType()))
opportunity_df=opportunity_df.withColumn("employee_id",col("employee_id").cast(IntegerType()))
opportunity_df=opportunity_df.withColumn("customer_id",col("customer_id").cast(IntegerType()))
opportunity_df=opportunity_df.withColumn("product_id",col("product_id").cast(IntegerType()))

In [0]:
opportunity_df=parse_mixed_date(opportunity_df,"start_date")
opportunity_df=parse_mixed_date(opportunity_df,"end_date")

In [0]:
opportunity_df=opportunity_df.withColumn("start_date",col("start_date").cast(DateType()))
opportunity_df=opportunity_df.withColumn("end_date",col("end_date").cast(DateType()))

### Removing Unnecessary Symbols

In [0]:
opportunity_df=opportunity_df.withColumn(
    "revenue_amount",
    regexp_replace(col("revenue_amount"), r"[£$€,]", "").cast("double")
)

In [0]:
opportunity_df=opportunity_df.withColumn("Month",date_trunc('month',col("start_date")).cast(DateType()))

## Country Master Table

In [0]:
country_master_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_country_master`""")

## fx_rate Table

In [0]:
fx_rate_df=spark.read.table(f"""{BRONZE_SCHEMA_PATH}.`bronze_fx_rate`""")

### TypeCasting

In [0]:
fx_rate_df=fx_rate_df.withColumn("fx_rate_to_gbp",col("fx_rate_to_gbp").cast(DoubleType()))

In [0]:
fx_rate_df=parse_mixed_date(fx_rate_df,"effective_date")
fx_rate_df=fx_rate_df.withColumn("effective_date",col("effective_date").cast(DateType()))

# Business Logics

In [0]:
opportunity_df=opportunity_df.alias("o").\
    join(customer_df.alias("c"), 
        on=col("o.customer_id") == col("c.customer_id"), 
        how="left")\
    .select("o.*", "c.country").alias("co")\
    .join(country_master_df.alias("cm"),
        on=col("co.country") == col("cm.country_code"), 
        how="left")\
    .select("co.*", "cm.currency_code").alias("coc")\
    .join(fx_rate_df.alias("fx"),
        on=col("coc.currency_code") == col("fx.currency_code"),
        how="left")\
    .select("coc.*", "fx.fx_rate_to_gbp")

In [0]:
opportunity_df=opportunity_df.withColumn("revenue_in_gpb",col("revenue_amount")*col("fx_rate_to_gbp"))

In [0]:
opportunity_df = opportunity_df.withColumn(
    "contract_months",
    when(col("contract_term") == "Monthly", 1)
    .when(col("contract_term") == "Yearly", 12)
)
opportunity_df = opportunity_df.withColumn(
    "mrr_in_gpb",
    col("revenue_in_gpb") / col("contract_months")
)

In [0]:

events_df = opportunity_df.withColumn(
    "event_type",
    when(col("close_status") == "Won", "NEW")
    .when(col("close_status") == "Lost", "CHURN")
    .otherwise(None)
)
events_df=events_df.filter(col("event_type").isNotNull())

# Creating Schema

In [0]:
spark.sql(f"""create schema if not exists {SILVER_SCHEMA_PATH}""")

# Saving Dataframe

In [0]:
country_master_df.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA_PATH}.`silver_country_master`")
customer_df.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA_PATH}.`silver_customer`")
employee_df.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA_PATH}.`silver_employee`")
opportunity_df.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA_PATH}.`silver_opportunity`")
product_df.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA_PATH}.`silver_product`")
events_df.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA_PATH}.`silver_events`")
fx_rate_df.write.mode("overwrite").saveAsTable(f"{SILVER_SCHEMA_PATH}.`silver_fx_rate_df`")